# 🛒 Nghiên cứu Đánh giá Chất lượng Dịch vụ WinMart theo Mô hình SERVQUAL
## 📄 Notebook 01: Quy trình Làm sạch Dữ liệu & Chuẩn hóa Biến (Data Cleaning Pipeline)

**Mục tiêu của Notebook**:
1. Khám phá dữ liệu khảo sát thô từ Google Forms (`data_servqual.xlsx`).
2. Đối chiếu trực tiếp với Bảng từ điển dữ liệu (`codebook_servqual_mapping.xlsx`).
3. Làm sạch, chuẩn hóa tên cột tiếng Anh (`snake_case`) và các biến nhân khẩu học.
4. Sàng lọc chất lượng phiếu khảo sát (loại bỏ phiếu không phải sinh viên, phiếu thiếu câu hỏi, phiếu đánh toàn điểm 5, straightlining).
5. Tính toán sơ bộ điểm Cảm nhận ($P$), Kỳ vọng ($E$) và Khoảng cách Dịch vụ ($Gap = P - E$).
6. Cung cấp khu vực code mở rộng để bạn tùy biến phân tích thêm.

### 1. Import các thư viện phân tích & Module lõi (`servqual_core`)

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import module lõi servqual_core
import servqual_core as core

# Thiết lập hiển thị bảng & đồ họa
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ Đã import thành công các thư viện và module servqual_core!')

### 2. Đọc dữ liệu khảo sát thô & Kiểm tra cấu trúc ban đầu

In [ ]:
raw_file_path = Path('data/data_servqual.xlsx')
df_raw = pd.read_excel(raw_file_path)

print(f'📊 Kích thước dữ liệu thô: {df_raw.shape[0]} dòng (phiếu khảo sát) x {df_raw.shape[1]} cột')
df_raw.head(3)

### 3. Tra cứu Từ điển Dữ liệu (Codebook Mapping)
Notebook liên kết trực tiếp với file Excel tra cứu `codebook_servqual_mapping.xlsx` để bạn dễ dàng đối chiếu tên cột cũ / mới.

In [ ]:
codebook_path = Path('data/codebook_servqual_mapping.xlsx')
if codebook_path.exists():
    df_codebook_cols = pd.read_excel(codebook_path, sheet_name='Column_Mapping')
    print(f'📖 Tổng số biến trong Codebook: {len(df_codebook_cols)}')
    display(df_codebook_cols[['clean_column_name', 'dimension', 'metric_type', 'data_type', 'description_vn']].head(10))
else:
    print('⚠️ Chưa tìm thấy file codebook_servqual_mapping.xlsx')

### 4. Thực thi Pipeline Làm sạch & Lọc Phiếu Khảo sát (Data Cleaning)

In [ ]:
# Thực hiện làm sạch dữ liệu (drop_straightline=False hoặc True)
clean_df, dropped_df, stats = core.clean_dataset(df_raw, drop_straightline=False)

print('=' * 55)
print('📋 KẾT QUẢ LÀM SẠCH DỮ LIỆU (DATA QUALITY REPORT)')
print('=' * 55)
print(f'Tổng số phiếu thu thập ban đầu : {stats["n_raw"]}')
print(f'Số phiếu hợp lệ (Clean)         : {stats["n_clean"]} ({stats["valid_rate"]:.1%})')
print(f'Số phiếu bị loại (Dropped)      : {stats["n_dropped"]}')
print('\nChi tiết lý do loại bỏ phiếu:')
for reason, count in stats['drop_breakdown'].items():
    print(f'  - {reason:<28}: {count} phiếu')
print('=' * 55)

### 5. Trực quan hóa Chẩn đoán Chất lượng Mẫu (Quality Diagnostics Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 1. Pie chart tỉ lệ hợp lệ
axes[0].pie(
    [stats['n_clean'], stats['n_dropped']],
    labels=['Hợp lệ (Clean)', 'Loại bỏ (Dropped)'],
    autopct='%1.1f%%',
    colors=['#2ca02c', '#d62728'],
    explode=(0.06, 0),
    startangle=140,
    textprops={'fontsize': 11}
)
axes[0].set_title('Tỉ lệ Phiếu Khảo sát Hợp lệ', fontsize=12, fontweight='bold', pad=10)

# 2. Bar chart lý do loại
if stats['n_dropped'] > 0:
    reasons_df = pd.DataFrame(list(stats['drop_breakdown'].items()), columns=['Reason', 'Count'])
    sns.barplot(data=reasons_df, x='Count', y='Reason', palette='Reds_r', ax=axes[1])
    axes[1].set_title('Phân bổ Lý do Loại bỏ Phiếu', fontsize=12, fontweight='bold', pad=10)
    axes[1].set_xlabel('Số lượng phiếu', fontsize=10)
    axes[1].set_ylabel('')
else:
    axes[1].text(0.5, 0.5, 'Không có phiếu nào bị loại', ha='center', va='center')

plt.tight_layout()
plt.show()

### 6. Xem trước Dữ liệu Sạch & Tính toán Sơ bộ Điểm SERVQUAL (P, E, Gap)

In [ ]:
# Tính toán điểm SERVQUAL theo biến và theo chiều
df_item_scores, df_dim_scores = core.calculate_servqual_scores(clean_df)

print('⭐ KẾT QUẢ KHOẢNG CÁCH CHẤT LƯỢNG (GAP = P - E) THEO 5 CHIỀU SERVQUAL:')
display(df_dim_scores.sort_values(by='gap_mean'))

print('\n⭐ TOP 5 BIẾN CÓ GAP ÂM LỚN NHẤT (CẦN CẢI THIỆN CẤP BÁCH):')
display(df_item_scores.sort_values(by='gap_mean')[['item_code', 'dimension_vn', 'p_mean', 'e_mean', 'gap_mean', 'gap_rank']].head(5))

### 7. Khu vực Tương tác & Phân tích Mở rộng (Custom Exploration Code)
Bạn có thể chủ động viết thêm code Python tại các ô dưới đây để truy vấn, lọc dữ liệu theo nhóm nhân khẩu học hoặc tính toán thêm:

In [ ]:
# Ví dụ 1: Xem phân bổ giới tính và hình thức cư trú trong tập dữ liệu sạch
print('--- Phân bổ Giới tính ---')
display(clean_df['gender'].value_counts(normalize=True).mul(100).round(2).to_frame(name='Tỷ lệ (%)'))

print('--- Phân bổ Nơi ở ---')
display(clean_df['residence'].value_counts(normalize=True).mul(100).round(2).to_frame(name='Tỷ lệ (%)'))

In [ ]:
# Ví dụ 2: Xem điểm Cảm nhận (P) trung bình theo từng nhóm Sinh hoạt phí
p_demo_summary = clean_df.groupby('monthly_budget')[['tangibles_1_p', 'reliability_1_p', 'responsiveness_1_p', 'assurance_1_p', 'empathy_1_p']].mean().round(2)
display(p_demo_summary)

In [ ]:
# [Khu vực dành cho bạn viết thêm code phân tích tự do]
# Ví dụ: clean_df.describe()

### 8. Xuất Dữ liệu Đã Làm sạch ra CSV & Excel

In [ ]:
output_clean_csv = Path('data/data_servqual_clean.csv')
output_dropped_csv = Path('data/data_servqual_dropped.csv')
output_clean_xlsx = Path('data/data_servqual_clean.xlsx')

# Xuất file CSV với encoding utf-8-sig (tránh lỗi font tiếng Việt khi mở bằng Excel)
clean_df.to_csv(output_clean_csv, index=False, encoding='utf-8-sig')
dropped_df.to_csv(output_dropped_csv, index=False, encoding='utf-8-sig')

# Xuất file Excel sạch kèm các sheet kết quả
with pd.ExcelWriter(output_clean_xlsx, engine='openpyxl') as writer:
    clean_df.to_excel(writer, sheet_name='Clean_Data', index=False)
    df_item_scores.to_excel(writer, sheet_name='Item_Scores', index=False)
    df_dim_scores.to_excel(writer, sheet_name='Dimension_Scores', index=False)

print('✅ Đã lưu thành công các file dữ liệu sạch:')
print(f'  - {output_clean_csv.name}')
print(f'  - {output_dropped_csv.name}')
print(f'  - {output_clean_xlsx.name}')